# Revenue Leakage Intelligence System
## Formal Business Analysis Notebook

**Business question:** What is driving revenue leakage, where is revenue at risk, and which customers or segments should the business prioritize first?

### Analytical workflow
1. Data loading & profiling
2. Headline metrics — the revenue funnel
3. Leakage driver decomposition
4. Segment analysis — who carries disproportionate risk?
5. Trend analysis — is leakage worsening?
6. Customer prioritization — who to act on first
7. Sales rep discount analysis
8. Concentration risk
9. Validation of key calculations
10. Executive summary & recommendations

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)

COLORS = ['#1B4F72', '#2E86C1', '#27AE60', '#E67E22', '#C0392B', '#8E44AD', '#F1C40F']

# Load raw data
customers = pd.read_csv('../data/raw/customers.csv', parse_dates=['onboard_date'])
revenue = pd.read_csv('../data/raw/monthly_revenue.csv')
revenue['month'] = pd.to_datetime(revenue['month'])
usage = pd.read_csv('../data/raw/product_usage.csv')

# Load processed data
scorecard = pd.read_csv('../data/processed/risk_scorecard.csv')
payments = pd.read_csv('../data/processed/payment_analysis.csv')
discounts_df = pd.read_csv('../data/processed/discount_analysis.csv')
rep_discounts = pd.read_csv('../data/processed/rep_discount_impact.csv')
deterioration = pd.read_csv('../data/processed/deterioration_analysis.csv')

print(f'Raw tables: customers={len(customers):,}, revenue={len(revenue):,}, usage={len(usage):,}')
print(f'Processed:  scorecard={len(scorecard):,}, payments={len(payments):,}')
print(f'Period:     {revenue["month"].min():%Y-%m} to {revenue["month"].max():%Y-%m} ({revenue["month"].nunique()} months)')

## 1. Data Profiling

Quick validation: row counts, null rates, key distributions.

In [ ]:
from src.data_profiler import profile_dataframe, print_profile

for name, df in [('customers', customers), ('monthly_revenue', revenue)]:
    profile = profile_dataframe(df, name)
    print_profile(profile)

In [ ]:
# Customer segment distribution and ACV shape
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

customers['segment'].value_counts().plot(kind='bar', ax=axes[0], color='#2E86C1')
axes[0].set_title('Customers by Segment')
axes[0].set_xlabel('')

customers.groupby('segment')['annual_contract_value'].median().sort_values().plot(
    kind='barh', ax=axes[1], color='#27AE60')
axes[1].set_title('Median ACV by Segment')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

customers['region'].value_counts().plot(kind='bar', ax=axes[2], color='#E67E22')
axes[2].set_title('Customers by Region')
axes[2].set_xlabel('')

for ax in axes:
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# Key profiling stats
print(f"Churn rate: {customers['is_churned'].mean()*100:.1f}%")
print(f"ACV range: ${customers['annual_contract_value'].min():,.0f} - ${customers['annual_contract_value'].max():,.0f}")
print(f"ACV skew: {customers['annual_contract_value'].skew():.2f} (right-skewed, typical for B2B SaaS)")

## 2. Headline Metrics — The Revenue Funnel

The core question: how much revenue leaks between contract and collection?

In [ ]:
# Revenue funnel quantification
total_contracted = revenue['mrr_contracted'].sum()
total_billed = revenue['mrr_billed'].sum()
total_collected = revenue['mrr_collected'].sum()
discount_leak = total_contracted - total_billed
collection_leak = total_billed - total_collected
total_leak = total_contracted - total_collected

print("═" * 65)
print("  REVENUE FUNNEL (24 months)")
print("═" * 65)
print(f"  Contracted revenue:    ${total_contracted:>12,.0f}")
print(f"  − Discount leakage:   ${discount_leak:>12,.0f}  ({discount_leak/total_contracted*100:.1f}%)")
print(f"  = Billed revenue:      ${total_billed:>12,.0f}")
print(f"  − Collection leakage: ${collection_leak:>12,.0f}  ({collection_leak/total_billed*100:.1f}%)")
print(f"  = Collected revenue:   ${total_collected:>12,.0f}")
print(f"  ─────────────────────────────────────")
print(f"  Total leakage:         ${total_leak:>12,.0f}  ({total_leak/total_contracted*100:.1f}%)")
print(f"═" * 65)

# Revenue funnel chart
fig, ax = plt.subplots(figsize=(10, 5))
stages = ['Contracted\nRevenue', 'After\nDiscounts', 'After\nCollection\nFailures']
values = [total_contracted, total_billed, total_collected]
colors_bar = [COLORS[0], COLORS[3], COLORS[4]]
bars = ax.bar(stages, values, color=colors_bar, width=0.5, edgecolor='white', linewidth=2)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + total_contracted*0.01,
            f'${val/1e6:.1f}M', ha='center', fontweight='bold', fontsize=12)
ax.set_title('Revenue Funnel: $6.0M Lost Between Contract and Collection',
             fontsize=14, fontweight='bold', pad=15)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))
ax.spines[['top', 'right']].set_visible(False)
ax.set_ylim(0, total_contracted * 1.12)
plt.tight_layout()
plt.show()

## 3. Leakage Driver Decomposition

Which risk dimension contributes most? Payment failures, deterioration, churn, or discounting?

In [ ]:
# Payment status breakdown — where does collection leakage come from?
ps = revenue.groupby('payment_status').agg(
    invoices=('customer_id', 'count'),
    billed=('mrr_billed', 'sum'),
    collected=('mrr_collected', 'sum'),
).reset_index()
ps['pct_invoices'] = ps['invoices'] / ps['invoices'].sum() * 100
ps['collection_rate'] = ps['collected'] / ps['billed'] * 100
ps['leakage'] = ps['billed'] - ps['collected']
print("Payment Status Breakdown:")
display(ps.sort_values('leakage', ascending=False))

# Weighted risk contribution by dimension
churn_w = scorecard['risk_score_churn'].mean() * 0.30
det_w = scorecard['risk_score_deterioration'].mean() * 0.25
disc_w = scorecard['risk_score_discount'].mean() * 0.20
pay_w = scorecard['risk_score_payment'].mean() * 0.25
total_w = churn_w + det_w + disc_w + pay_w

drivers = pd.DataFrame({
    'Driver': ['Payment Failures', 'MRR Deterioration', 'Silent Churn', 'Discount Anomalies'],
    'Weighted Points': [pay_w, det_w, churn_w, disc_w],
    'Pct of Total': [pay_w/total_w*100, det_w/total_w*100, churn_w/total_w*100, disc_w/total_w*100],
}).sort_values('Weighted Points', ascending=True)

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(drivers['Driver'], drivers['Weighted Points'], color=[COLORS[4], COLORS[3], COLORS[0], COLORS[5]])
ax.set_title('Payment Failures Are the #1 Revenue Leakage Driver (44% of Weighted Risk)',
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Weighted Risk Points')
for i, (_, r) in enumerate(drivers.iterrows()):
    ax.text(r['Weighted Points'] + 0.1, i, f"{r['Pct of Total']:.1f}%", va='center', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Segment Analysis — Who Carries Disproportionate Risk?

Risk Intensity = (% of total risk) / (% of total ARR). Values >1.0× indicate a segment carries more risk than its revenue share would predict.

In [ ]:
# Segment risk analysis
seg = scorecard.groupby('segment').agg(
    customers=('customer_id', 'count'),
    total_acv=('annual_contract_value', 'sum'),
    total_at_risk=('revenue_at_risk', 'sum'),
    avg_risk=('composite_risk_score', 'mean'),
    avg_churn=('risk_score_churn', 'mean'),
    avg_deterioration=('risk_score_deterioration', 'mean'),
    avg_discount=('risk_score_discount', 'mean'),
    avg_payment=('risk_score_payment', 'mean'),
).reset_index()
seg['pct_arr'] = seg['total_acv'] / seg['total_acv'].sum() * 100
seg['pct_risk'] = seg['total_at_risk'] / seg['total_at_risk'].sum() * 100
seg['risk_intensity'] = seg['pct_risk'] / seg['pct_arr']

print("Segment Risk Summary:")
display(seg[['segment', 'customers', 'total_acv', 'pct_arr', 'total_at_risk', 'pct_risk', 'risk_intensity', 'avg_risk']])

# Risk heatmap by dimension
fig, ax = plt.subplots(figsize=(10, 4))
heat_data = seg.set_index('segment')[['avg_churn', 'avg_deterioration', 'avg_discount', 'avg_payment']]
heat_data.columns = ['Churn', 'Deterioration', 'Discount', 'Payment']
sns.heatmap(heat_data, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, linewidths=1,
            cbar_kws={'label': 'Avg Risk Score'})
ax.set_title('Payment Risk Dominates Across All Segments', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Segment × Region hotspot
seg_region = scorecard.groupby(['segment', 'region']).agg(
    at_risk=('revenue_at_risk', 'sum'),
    acv=('annual_contract_value', 'sum'),
).reset_index()
seg_region['risk_rate'] = seg_region['at_risk'] / seg_region['acv'] * 100

pivot = seg_region.pivot_table(index='segment', columns='region', values='risk_rate')
print("\nRisk Rate (%) by Segment × Region:")
display(pivot.round(1))
hotspot = seg_region.loc[seg_region['risk_rate'].idxmax()]
print(f"\n⚠️ Hotspot: {hotspot['segment']} × {hotspot['region']} — {hotspot['risk_rate']:.1f}% at risk")

## 5. Trend Analysis — Is Leakage Worsening?

Comparing the first 6 months vs the last 6 months of the analysis period.

In [ ]:
# Monthly trend computation
monthly = revenue.groupby('month').agg(
    contracted=('mrr_contracted', 'sum'),
    billed=('mrr_billed', 'sum'),
    collected=('mrr_collected', 'sum'),
    customers=('customer_id', 'nunique'),
    avg_engagement=('engagement_score', 'mean'),
    avg_discount=('discount_pct', 'mean'),
    failed_pct=('payment_status', lambda x: (x == 'failed').mean() * 100),
).reset_index()
monthly['collection_rate'] = monthly['collected'] / monthly['billed'] * 100
monthly['total_leakage_rate'] = (monthly['contracted'] - monthly['collected']) / monthly['contracted'] * 100

# First 6 vs last 6
first_6 = monthly.head(6)
last_6 = monthly.tail(6)

print("Trend Comparison: First 6 Months vs Last 6 Months")
print("─" * 60)
metrics = {
    'Collection rate (%)': (first_6['collection_rate'].mean(), last_6['collection_rate'].mean()),
    'Avg engagement': (first_6['avg_engagement'].mean(), last_6['avg_engagement'].mean()),
    'Total leakage rate (%)': (first_6['total_leakage_rate'].mean(), last_6['total_leakage_rate'].mean()),
    'Active customers': (first_6['customers'].mean(), last_6['customers'].mean()),
    'Failed payment (%)': (first_6['failed_pct'].mean(), last_6['failed_pct'].mean()),
}
for name, (early, late) in metrics.items():
    change = late - early
    print(f"  {name:28s}  {early:8.1f} → {late:8.1f}  ({change:+.1f})")

# 4-panel trend chart
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0,0].plot(monthly['month'], monthly['collection_rate'], color=COLORS[0], linewidth=2.5, marker='o', markersize=4)
axes[0,0].axhline(monthly['collection_rate'].mean(), color=COLORS[4], linestyle='--', alpha=0.5)
axes[0,0].set_title('Collection Rate Shows Structural Decline', fontweight='bold')
axes[0,0].set_ylabel('Collection Rate (%)')
axes[0,0].set_ylim(80, 95)

axes[0,1].plot(monthly['month'], monthly['avg_engagement'], color=COLORS[1], linewidth=2.5, marker='o', markersize=4)
z = np.polyfit(range(len(monthly)), monthly['avg_engagement'], 1)
axes[0,1].plot(monthly['month'], np.polyval(z, range(len(monthly))), '--', color=COLORS[4], alpha=0.7,
               label=f'Trend: {z[0]:+.2f}/mo')
axes[0,1].set_title('Engagement Declining Across the Portfolio', fontweight='bold')
axes[0,1].set_ylabel('Avg Engagement Score')
axes[0,1].legend(fontsize=9)

axes[1,0].bar(monthly['month'], (monthly['contracted'] - monthly['billed'])/1000, color=COLORS[3], alpha=0.7, width=20)
axes[1,0].set_title('Discount Leakage Stable Over Time', fontweight='bold')
axes[1,0].set_ylabel('Discount Leakage ($K)')

axes[1,1].stackplot(monthly['month'],
                     (monthly['contracted'] - monthly['billed'])/1000,
                     (monthly['billed'] - monthly['collected'])/1000,
                     labels=['Discount', 'Collection Gap'], colors=[COLORS[3], COLORS[4]], alpha=0.7)
axes[1,1].set_title('Total Monthly Leakage Composition', fontweight='bold')
axes[1,1].set_ylabel('Leakage ($K)')
axes[1,1].legend(fontsize=9, loc='upper left')

for ax in axes.flat:
    ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 6. Customer Prioritization — Who to Act On First

Rank customers by revenue-at-risk and group into actionable intervention cohorts.

In [ ]:
# Top 20 accounts by revenue at risk
top20 = scorecard.nlargest(20, 'revenue_at_risk')
total_risk = scorecard['revenue_at_risk'].sum()
top20_risk = top20['revenue_at_risk'].sum()

print(f"Top 20 accounts: ${top20_risk:,.0f} of ${total_risk:,.0f} total risk ({top20_risk/total_risk*100:.1f}%)")
print(f"→ 4% of customers account for {top20_risk/total_risk*100:.0f}% of revenue risk\n")
display(top20[['customer_id', 'segment', 'annual_contract_value', 'composite_risk_score',
               'risk_tier', 'revenue_at_risk', 'risk_score_churn', 'risk_score_payment']])

# Actionable cohorts
cohort1 = scorecard[(scorecard['segment'].isin(['Enterprise', 'Mid-Market'])) &
                     (scorecard['composite_risk_score'] > 30) &
                     (scorecard['avg_engagement'].fillna(0) < 40)]
cohort2 = scorecard[scorecard['risk_score_payment'] > 60]
cohort3 = scorecard[scorecard['risk_score_discount'] > 20]
cohort4 = scorecard[scorecard['risk_score_deterioration'] > 50]

print("\nActionable Intervention Cohorts:")
print("─" * 80)
for name, df, action, owner in [
    ('High-Value Disengaging', cohort1, 'CSM outreach + QBR', 'Retention'),
    ('Payment Failures', cohort2, 'Collections escalation', 'Finance/AR'),
    ('Over-Discounted', cohort3, 'Discount policy review', 'Sales Ops'),
    ('Rapid Deteriorators', cohort4, 'Product adoption program', 'CSM + Product'),
]:
    print(f"  {name:28s}  {len(df):>4d} customers  ${df['revenue_at_risk'].sum():>10,.0f} at risk  → {action} ({owner})")

In [ ]:
# Prioritization matrix: engagement vs ACV, sized by risk
fig, ax = plt.subplots(figsize=(11, 7))
sc_plot = scorecard.dropna(subset=['avg_engagement']).copy()
tier_colors = {'Low': '#27AE60', 'Medium': '#F1C40F', 'High': '#E67E22', 'Critical': '#C0392B'}
for tier in ['Low', 'Medium', 'High', 'Critical']:
    mask = sc_plot['risk_tier'] == tier
    if mask.sum() == 0:
        continue
    ax.scatter(sc_plot.loc[mask, 'avg_engagement'],
               sc_plot.loc[mask, 'annual_contract_value'] / 1000,
               c=tier_colors[tier], label=tier, alpha=0.6,
               s=sc_plot.loc[mask, 'revenue_at_risk'] / 100 + 20,
               edgecolors='white', linewidth=0.5)

ax.axvline(40, color='gray', linestyle=':', alpha=0.5)
ax.axhline(sc_plot['annual_contract_value'].median() / 1000, color='gray', linestyle=':', alpha=0.5)
ax.annotate('PRIORITY\nINTERVENTION', xy=(12, sc_plot['annual_contract_value'].quantile(0.8)/1000),
            fontsize=13, fontweight='bold', color=COLORS[4], alpha=0.7)

ax.set_title('Low Engagement × High ACV = Immediate Action Required',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Engagement Score (higher = healthier)')
ax.set_ylabel('Annual Contract Value ($K)')
ax.legend(title='Risk Tier', loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 7. Validation of Key Calculations

Every number presented to stakeholders must be verifiable.

In [ ]:
# Validation checks
print("Validation Checks")
print("═" * 60)

# V1: Revenue funnel arithmetic
v1_lhs = total_contracted - discount_leak - collection_leak
v1 = abs(v1_lhs - total_collected) < 1
print(f"{'✅' if v1 else '❌'} Revenue funnel: contracted - discounts - collection gaps = collected")
print(f"    ${v1_lhs:,.0f} vs ${total_collected:,.0f}")

# V2: Segment ARR = total ARR
seg_sum = seg['total_acv'].sum()
total_arr = customers['annual_contract_value'].sum()
v2 = abs(seg_sum - total_arr) < 1
print(f"{'✅' if v2 else '❌'} Segment ARR sums to total: ${seg_sum:,.0f} vs ${total_arr:,.0f}")

# V3: Score bounds
for col in ['risk_score_churn', 'risk_score_deterioration', 'risk_score_discount',
            'risk_score_payment', 'composite_risk_score']:
    oob = ((scorecard[col] < 0) | (scorecard[col] > 100)).sum()
    print(f"{'✅' if oob == 0 else '❌'} {col} in [0,100]: {oob} out of bounds")

# V4: Composite = weighted sum
recomputed = (scorecard['risk_score_churn'] * 0.30 + scorecard['risk_score_deterioration'] * 0.25 +
              scorecard['risk_score_discount'] * 0.20 + scorecard['risk_score_payment'] * 0.25).round(1)
max_diff = (scorecard['composite_risk_score'] - recomputed).abs().max()
print(f"{'✅' if max_diff < 0.2 else '❌'} Composite = weighted sum (max diff: {max_diff:.4f})")

# V5: revenue_at_risk <= ACV
v5 = (scorecard['revenue_at_risk'] > scorecard['annual_contract_value']).sum()
print(f"{'✅' if v5 == 0 else '❌'} revenue_at_risk ≤ ACV: {v5} violations")

# V6: No negative billed/collected
v6 = (revenue['mrr_billed'] < 0).sum() + (revenue['mrr_collected'] < 0).sum()
print(f"{'✅' if v6 == 0 else '❌'} No negative billed/collected: {v6} violations")

## 8. Executive Summary

### The Bottom Line
The company is losing **$6.0M (18.2%)** of contracted revenue through collection failures ($4.5M) and discounting ($1.5M). An additional **$4.2M in ARR (21.7%)** is at risk based on customer health scoring.

### Key Findings
1. **Collection failures are the #1 leakage driver** — 18% of invoices result in incomplete collection (failed, partial, or pending). Payment risk contributes 44% of the composite risk score.
2. **Payment risk dominates across all segments** — Enterprise, Mid-Market, and SMB all show payment as the highest-scoring risk dimension.
3. **Revenue risk is concentrated** — the top 20 accounts (4% of customers) represent 25% of total revenue at risk. Targeted intervention on a small number of accounts can address the majority of risk.
4. **Engagement is declining** — average engagement dropped from 66.7 to 52.0 over the analysis period. "Gradual decline" customers show the steepest erosion.
5. **Leakage is worsening** — collection rate fell 1.5pp, total leakage rate rose 1.1pp, and 67 fewer customers are active vs. the start of the period.

### Recommendations
| Priority | Action | Accounts | ACV at Risk | Owner |
|----------|--------|----------|-------------|-------|
| Immediate | Escalate top 20 risk accounts to CSM leadership | 20 | $1.0M | Retention |
| Immediate | Collections escalation on payment failures | 79 | $1.0M | Finance/AR |
| Short-term | Discount guardrails (flag >25% for approval) | 85 | $465K | Sales Ops |
| Short-term | Re-engagement program for disengaging accounts | 51 | $1.3M | CSM + Product |
| Strategic | Build NRR tracking by segment and cohort | — | — | Rev Ops |
| Strategic | Develop predictive churn model | — | — | Data Science |

### Caveats
- Synthetic data — patterns are realistic but generated
- Risk weights are judgement-based, not empirically calibrated
- Scores indicate risk association, not causation
- Point-in-time analysis without predictive modeling

## 9. Publication-Quality Visualization Pack

The following 10 charts provide a comprehensive visual narrative of revenue leakage. Each chart uses a consistent design system with professional typography and colour palette, suitable for executive presentation and portfolio demonstration.

| # | Chart | Type | Purpose |
|---|-------|------|---------|
| 01 | Revenue Erosion Trend | Layered area | Shows gap between contracted, billed, and collected revenue |
| 02 | Risk Composition | Stacked area | Decomposition of leakage risk by channel over time |
| 03 | Risk Drivers by Segment | Grouped horizontal bar | Compares risk dimensions across Enterprise/Mid-Market/SMB |
| 04 | Top 25 at Risk | Lollipop | Prioritized customer intervention list |
| 05 | Segment Mix | Paired donut | ARR vs Risk distribution reveals concentration |
| 06 | Cohort Retention | Heatmap | Retention by onboarding quarter |
| 07 | Payment Anomalies | Dual-axis bar+line | Payment failures correlated with engagement decline |
| 08 | Prioritization Matrix | Bubble scatter | Risk score × Revenue at Risk with quadrant labels |
| 09 | Collection Distribution | Violin + strip | Variance in collection rates by segment |
| 10 | Rep Discount Impact | Horizontal bar | Revenue lost to discounting per sales rep |

In [ ]:
from IPython.display import Image, display
import os

chart_dir = '../outputs/charts/publication'
charts = [
    ('01_revenue_erosion_trend.png', 'Chart 01 — Revenue Erosion: Contract to Collection'),
    ('02_risk_composition_over_time.png', 'Chart 02 — Leakage Risk Composition Over Time'),
    ('03_risk_drivers_by_segment.png', 'Chart 03 — Risk Driver Intensity by Segment'),
    ('04_top25_revenue_at_risk.png', 'Chart 04 — Top 25 Customers by Revenue at Risk'),
    ('05_segment_mix_donuts.png', 'Chart 05 — Segment Mix: Revenue vs Risk Concentration'),
    ('06_cohort_retention_heatmap.png', 'Chart 06 — Cohort Revenue Retention Heatmap'),
    ('07_payment_anomaly_trend.png', 'Chart 07 — Payment Anomalies & Engagement Trajectory'),
    ('08_prioritization_matrix.png', 'Chart 08 — Customer Prioritization Matrix'),
    ('09_collection_rate_distribution.png', 'Chart 09 — Collection Rate Distribution by Segment'),
    ('10_rep_discount_impact.png', 'Chart 10 — Sales Rep Discount Revenue Impact'),
]

for filename, title in charts:
    path = os.path.join(chart_dir, filename)
    if os.path.exists(path):
        print(f'\n{"="*70}')
        print(f'  {title}')
        print(f'{"="*70}')
        display(Image(filename=path, width=900))
    else:
        print(f'⚠ Not found: {path}')

### Chart Design Rationale

**Design system choices:**
- **Colour palette**: Professional dark-on-light scheme with semantic colours (red=danger, green=success, amber=warning, blue=primary)
- **Typography**: Helvetica Neue at consistent sizes (18pt titles, 11pt subtitles, 9pt annotations)
- **Grid**: Light grey gridlines at 40% opacity to guide the eye without competing with data
- **Annotations**: Every chart includes contextual annotations that explain "so what?" rather than just "what"

**Chart type rationale:**
- **Area charts** (01, 02) chosen for magnitude over time — the filled area emphasises the *volume* of leakage
- **Lollipop** (04) preferred over bar chart for ranked lists — cleaner, reduces ink-to-data ratio
- **Heatmap** (06) is the natural choice for cohort × time retention matrices
- **Violin + strip** (09) shows both distribution shape *and* individual observations — richer than boxplot alone
- **Bubble scatter** (08) encodes 4 dimensions simultaneously (x, y, size, colour)

**Interpretation caveats:**
1. All data is synthetic — patterns are realistic but generated for demonstration
2. Risk scores are judgement-weighted, not empirically calibrated
3. Correlation (e.g., engagement vs payment failures) does not imply causation
4. Point-in-time snapshot — predictive modeling would strengthen forward-looking recommendations